<a href="https://colab.research.google.com/github/aayushbhurtel/MachineLearning/blob/main/SVM_Fraud_Detection_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SVM Fraud Detection Model – Week 6 Assignment
**Dataset:** Week6-Fraud_data (FraudTrain + FraudTest sheets)  
**Model:** Support Vector Machine (SVM) via Vaimal Excel Add-in  
**Target Variable:** `is_fraud`


## Setup – Install & Import Libraries

In [ ]:
# Install required libraries
!pip install pandas numpy matplotlib seaborn openpyxl -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from google.colab import files

print('Libraries loaded successfully!')

## Step 1 – Upload & Load Data

In [ ]:
# Upload the Excel file
print('Please upload Week6-Fraud_data.xlsx')
uploaded = files.upload()

In [ ]:
# Load FraudTrain and FraudTest sheets
filename = list(uploaded.keys())[0]
print(f'Loading: {filename}')

train_df = pd.read_excel(filename, sheet_name='fraudTrain')
test_df  = pd.read_excel(filename, sheet_name='fraudTest')

print(f'FraudTrain shape: {train_df.shape}')
print(f'FraudTest  shape: {test_df.shape}')
train_df.head(3)

In [ ]:
# Column overview
print('Columns:', list(train_df.columns))
print('\nData types:')
print(train_df.dtypes)

---
## Part (a) – Descriptive Measures of Column `amt` & Transaction Counts

In [ ]:
# ── Descriptive statistics for 'amt' (FraudTrain) ──
amt = train_df['amt']

stats = {
    'Count'          : len(amt),
    'Min'            : amt.min(),
    'Max'            : amt.max(),
    'Mean'           : round(amt.mean(), 4),
    'Std. Deviation' : round(amt.std(),  4),
}

stats_df = pd.DataFrame(stats.items(), columns=['Metric', 'Value'])
print('═' * 40)
print('  Descriptive Measures of amt Column')
print('═' * 40)
print(stats_df.to_string(index=False))
print()

In [ ]:
# ── Transaction counts ──
fraud_counts = train_df['is_fraud'].value_counts().sort_index()
normal_count    = int(fraud_counts.get(0, 0))
fraudulent_count = int(fraud_counts.get(1, 0))
total           = normal_count + fraudulent_count

counts_df = pd.DataFrame({
    'Transaction Type': ['Normal (is_fraud = 0)', 'Fraudulent (is_fraud = 1)', 'Total'],
    'Count'           : [normal_count, fraudulent_count, total],
    'Percentage'      : [
        f'{normal_count/total*100:.2f}%',
        f'{fraudulent_count/total*100:.2f}%',
        '100.00%'
    ]
})

print('═' * 50)
print('  Count of Fraudulent vs Normal Transactions')
print('═' * 50)
print(counts_df.to_string(index=False))

---
## Part (b) – Normalized Frequency Graphs: `category` and `is_fraud`

In [ ]:
# ── Compute normalized frequency per category ──
cat_fraud = (
    train_df.groupby(['category', 'is_fraud'])
    .size()
    .unstack(fill_value=0)
)
cat_fraud.columns = ['Normal', 'Fraud']
cat_fraud['Total']           = cat_fraud['Normal'] + cat_fraud['Fraud']
cat_fraud['Normal_Norm']     = (cat_fraud['Normal'] / cat_fraud['Total']).round(6)
cat_fraud['Fraud_Norm']      = (cat_fraud['Fraud']  / cat_fraud['Total']).round(6)

print('Normalized Frequency by Category:')
print(cat_fraud[['Normal_Norm','Fraud_Norm']].to_string())

In [ ]:
# ── Graph 1: Normalized frequency by category (grouped bar) ──
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle('Normalized Frequency by Category and Is_Fraud\n(FraudTrain Dataset)',
             fontsize=14, fontweight='bold')

cats = cat_fraud.index.tolist()
x    = np.arange(len(cats))
w    = 0.35

ax1 = axes[0]
ax1.bar(x - w/2, cat_fraud['Normal_Norm'], w, label='Normal (is_fraud=0)',
        color='steelblue', alpha=0.85, edgecolor='navy')
ax1.bar(x + w/2, cat_fraud['Fraud_Norm'],  w, label='Fraudulent (is_fraud=1)',
        color='crimson',   alpha=0.85, edgecolor='darkred')
ax1.set_xticks(x)
ax1.set_xticklabels([c.replace('_','\n') for c in cats], rotation=45, ha='right', fontsize=8)
ax1.set_ylabel('Normalized Frequency')
ax1.set_title('Normal vs. Fraudulent by Category')
ax1.legend(); ax1.set_ylim(0, 1.1); ax1.grid(axis='y', alpha=0.3)

# Graph 2: Fraud rate zoomed in
ax2 = axes[1]
colors = plt.cm.RdYlGn_r(np.linspace(0.2, 0.9, len(cats)))
bars = ax2.bar(x, cat_fraud['Fraud_Norm'] * 100, color=colors, edgecolor='black', alpha=0.85)
for bar, val in zip(bars, cat_fraud['Fraud_Norm']):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{val*100:.2f}%', ha='center', va='bottom', fontsize=7.5, fontweight='bold')
ax2.set_xticks(x)
ax2.set_xticklabels([c.replace('_','\n') for c in cats], rotation=45, ha='right', fontsize=8)
ax2.set_ylabel('Fraud Rate (%)')
ax2.set_title('Fraud Rate (%) by Category')
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('graph_category_fraud_frequency.png', dpi=150, bbox_inches='tight')
plt.show()
print('Graph saved!')

In [ ]:
# ── Graph 2: is_fraud distribution (Pie + Bar) ──
fig2, axes2 = plt.subplots(1, 2, figsize=(13, 6))
fig2.suptitle('Transaction Class Distribution (is_fraud)\nFraudTrain Dataset', fontsize=13, fontweight='bold')

labels_pie = [
    f'Normal (0)\n{normal_count:,}\n({normal_count/total*100:.2f}%)',
    f'Fraudulent (1)\n{fraudulent_count:,}\n({fraudulent_count/total*100:.2f}%)'
]
axes2[0].pie([normal_count, fraudulent_count], explode=(0, 0.1),
             labels=labels_pie, colors=['steelblue','crimson'],
             autopct='%1.2f%%', shadow=True, startangle=90)
axes2[0].set_title('Class Distribution (Pie)')

axes2[1].bar(['Normal (0)', 'Fraudulent (1)'],
             [normal_count, fraudulent_count],
             color=['steelblue','crimson'], edgecolor='black', alpha=0.85)
axes2[1].set_yscale('log')
axes2[1].set_ylabel('Count (log scale)')
axes2[1].set_title('Class Count (Log Scale)')
axes2[1].grid(axis='y', alpha=0.3)
for i, cnt in enumerate([normal_count, fraudulent_count]):
    axes2[1].text(i, cnt * 1.15, f'{cnt:,}', ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('graph_isfraud_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Data Preprocessing (for SVM Input)
Encoding categorical variables and selecting features

In [ ]:
# ── Encode categorical variables ──
from sklearn.preprocessing import LabelEncoder, StandardScaler

# Selected input features (drop non-numeric / high-cardinality columns)
DROP_COLS = ['Unnamed: 0', 'trans_date_trans_time', 'cc_num', 'first', 'last',
             'street', 'city', 'job', 'dob', 'trans_num', 'Data_FLAG']

CATEGORICAL_COLS = ['merchant', 'category', 'gender', 'state']

def preprocess(df, encoders=None, fit=True):
    df = df.drop(columns=[c for c in DROP_COLS if c in df.columns], errors='ignore')
    df = df.copy()
    if encoders is None:
        encoders = {}
    for col in CATEGORICAL_COLS:
        if col not in df.columns:
            continue
        if fit:
            le = LabelEncoder()
            df[col] = le.fit_transform(df[col].astype(str))
            encoders[col] = le
        else:
            le = encoders[col]
            # Handle unseen labels
            known = set(le.classes_)
            df[col] = df[col].apply(lambda x: x if x in known else le.classes_[0])
            df[col] = le.transform(df[col].astype(str))
    return df, encoders

train_proc, encoders = preprocess(train_df, fit=True)
test_proc,  _        = preprocess(test_df,  encoders=encoders, fit=False)

print('Preprocessed train shape:', train_proc.shape)
print('Preprocessed test  shape:', test_proc.shape)
print('Columns:', list(train_proc.columns))

In [ ]:
# ── Split features / target ──
TARGET = 'is_fraud'

X_train = train_proc.drop(columns=[TARGET])
y_train = train_proc[TARGET]
X_test  = test_proc.drop(columns=[TARGET])
y_test  = test_proc[TARGET]

# Standardize (important for SVM)
scaler  = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'X_train: {X_train_sc.shape}, y_train: {y_train.shape}')
print(f'X_test : {X_test_sc.shape},  y_test : {y_test.shape}')

---
## SVM Model Training
> ⚠️ **Note:** The full dataset has 1.6M rows. Training on all rows can take a long time.  
> We use a **stratified sample** (10,000 rows) for fast training. Change `SAMPLE_SIZE` to use more.

In [ ]:
from sklearn.svm import SVC
from sklearn.utils import resample

SAMPLE_SIZE = 10_000  # increase if you have time/RAM

# Stratified sample
train_sample = train_proc.groupby(TARGET, group_keys=False).apply(
    lambda x: x.sample(min(len(x), int(SAMPLE_SIZE * len(x)/len(train_proc))), random_state=42)
)
print(f'Sample size: {len(train_sample)}')
print('Sample class distribution:')
print(train_sample[TARGET].value_counts())

Xs = scaler.transform(train_sample.drop(columns=[TARGET]))
ys = train_sample[TARGET]

In [ ]:
# Train SVM (Linear kernel — same as Vaimal default)
print('Training SVM (Linear kernel)...')
svm = SVC(kernel='linear', C=1.0, probability=True, random_state=42)
svm.fit(Xs, ys)
print('Training complete!')

---
## Part (c) – Error Analysis: Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

# Test on a sample of test data (for speed)
TEST_SAMPLE = 8_507  # matches Vaimal result size
test_sample = test_proc.groupby(TARGET, group_keys=False).apply(
    lambda x: x.sample(min(len(x), int(TEST_SAMPLE * len(x)/len(test_proc))), random_state=42)
)
X_ts = scaler.transform(test_sample.drop(columns=[TARGET]))
y_ts = test_sample[TARGET]

y_pred = svm.predict(X_ts)

cm = confusion_matrix(y_ts, y_pred)
TN, FP, FN, TP = cm.ravel()

print('═' * 45)
print('       CONFUSION MATRIX')
print('═' * 45)
print(f'  True  Negatives (TN): {TN:>6,}  ✅')
print(f'  False Positives (FP): {FP:>6,}  ❌')
print(f'  False Negatives (FN): {FN:>6,}  ❌')
print(f'  True  Positives (TP): {TP:>6,}  ✅')
print('─' * 45)

precision = TP / (TP + FP) if (TP + FP) > 0 else 0
recall    = TP / (TP + FN) if (TP + FN) > 0 else 0
accuracy  = (TP + TN) / (TP + FP + TN + FN)
f1        = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

print(f'  Precision (P) = TP/(TP+FP):          {precision:.4f}  ({precision*100:.2f}%)')
print(f'  Recall    (R) = TP/(TP+FN):          {recall:.4f}  ({recall*100:.2f}%)')
print(f'  Accuracy      = (TP+TN)/Total:       {accuracy:.4f}  ({accuracy*100:.2f}%)')
print(f'  F1 Score      = 2*P*R/(P+R):         {f1:.4f}  ({f1*100:.2f}%)')

In [ ]:
# ── Visualize Confusion Matrix ──
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('SVM Model – Confusion Matrix & Performance Metrics', fontsize=13, fontweight='bold')

# Heatmap
cm_arr = np.array([[TN, FP], [FN, TP]])
im = axes[0].imshow(cm_arr, cmap='Blues')
axes[0].set_xticks([0,1]); axes[0].set_yticks([0,1])
axes[0].set_xticklabels(['Pred: Normal','Pred: Fraud'])
axes[0].set_yticklabels(['Actual: Normal','Actual: Fraud'])
labels_cm = [[f'TN={TN:,}\n(True Neg)', f'FP={FP:,}\n(False Pos)'],
             [f'FN={FN:,}\n(False Neg)', f'TP={TP:,}\n(True Pos)']]
thresh = cm_arr.max() / 2
for i in range(2):
    for j in range(2):
        axes[0].text(j, i, labels_cm[i][j], ha='center', va='center',
                    color='white' if cm_arr[i,j]>thresh else 'black',
                    fontsize=10, fontweight='bold')
axes[0].set_title('Confusion Matrix'); plt.colorbar(im, ax=axes[0])

# Metrics bar
metrics = ['Precision\nP=TP/(TP+FP)', 'Recall\nR=TP/(TP+FN)', 'Accuracy\n(TP+TN)/Total', 'F1 Score\n2PR/(P+R)']
values  = [precision, recall, accuracy, f1]
colors_b = ['#2196F3','#FF9800','#4CAF50','#9C27B0']
bars = axes[1].bar(metrics, values, color=colors_b, edgecolor='black', alpha=0.85)
for bar, val in zip(bars, values):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                f'{val:.4f}', ha='center', va='bottom', fontsize=11, fontweight='bold')
axes[1].set_ylim(0, 1.15); axes[1].grid(axis='y', alpha=0.3)
axes[1].set_title('Performance Metrics')
axes[1].axhline(0.9, color='red', linestyle='--', alpha=0.4, label='90%')
axes[1].legend()

plt.tight_layout()
plt.savefig('graph_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Part (d) – Single Transaction Prediction
Predict whether the given transaction is Fraud or Normal.

In [ ]:
# ── Input transaction features ──
# (Only numeric + encoded categorical features used by the model)

new_tx = {
    'merchant'  : 'fraud_Torphy-Goyette',
    'category'  : 'shopping_pos',
    'amt'       : 1318.89,
    'gender'    : 'M',
    'state'     : 'AK',
    'zip'       : 99921,
    'lat'       : 55.4732,
    'long'      : -133.1171,
    'city_pop'  : 1920,
    'unix_time' : 1386023256,
    'merch_lat' : 54.801713,
    'merch_long': -133.669108,
}

tx_df = pd.DataFrame([new_tx])

# Encode categoricals using fitted encoders
for col in CATEGORICAL_COLS:
    if col in tx_df.columns:
        le = encoders[col]
        val = str(tx_df[col].iloc[0])
        if val not in le.classes_:
            val = le.classes_[0]  # fallback
            print(f'  Note: {col}={tx_df[col].iloc[0]} not seen in training; using fallback.')
        tx_df[col] = le.transform([val])

# Align columns with training data
feature_cols = X_train.columns.tolist()
for col in feature_cols:
    if col not in tx_df.columns:
        tx_df[col] = 0
tx_df = tx_df[feature_cols]

# Scale and predict
tx_scaled = scaler.transform(tx_df)
prediction      = svm.predict(tx_scaled)[0]
pred_proba      = svm.predict_proba(tx_scaled)[0]
decision_value  = svm.decision_function(tx_scaled)[0]

print('═' * 50)
print('        TRANSACTION PREDICTION RESULT')
print('═' * 50)
print(f'  Predicted Class     : {"FRAUDULENT 🚨" if prediction == 1 else "NORMAL ✅"}')
print(f'  is_fraud value      : {prediction}')
print(f'  Probability (Normal): {pred_proba[0]:.4f} ({pred_proba[0]*100:.2f}%)')
print(f'  Probability (Fraud) : {pred_proba[1]:.4f} ({pred_proba[1]*100:.2f}%)')
print(f'  Decision Value      : {decision_value:.4f}')
print('─' * 50)
print(f'  Ground Truth        : FRAUDULENT (is_fraud = 1)')

In [ ]:
# ── Feature importance (linear SVM weights) ──
coef = svm.coef_[0]
feat_imp = pd.DataFrame({'Feature': feature_cols, 'Weight': coef})
feat_imp = feat_imp.reindex(feat_imp['Weight'].abs().sort_values(ascending=False).index)

plt.figure(figsize=(10, 5))
colors_w = ['crimson' if w > 0 else 'steelblue' for w in feat_imp['Weight']]
plt.barh(feat_imp['Feature'], feat_imp['Weight'], color=colors_w, edgecolor='black', alpha=0.85)
plt.axvline(0, color='black', linewidth=0.8)
plt.xlabel('SVM Weight')
plt.title('SVM Feature Weights (Linear Kernel)\nRed = positive (→ Fraud), Blue = negative (→ Normal)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('graph_feature_weights.png', dpi=150, bbox_inches='tight')
plt.show()
print(feat_imp.to_string(index=False))

---
## Summary of All Results

In [ ]:
print('=' * 60)
print(' SVM FRAUD DETECTION – COMPLETE RESULTS SUMMARY')
print('=' * 60)

print('\n(a) Descriptive Stats for amt (FraudTrain):')
print(f'    Count : {len(train_df):,}')
print(f'    Min   : ${train_df["amt"].min():.2f}')
print(f'    Max   : ${train_df["amt"].max():.2f}')
print(f'    Mean  : ${train_df["amt"].mean():.4f}')
print(f'    Std   : ${train_df["amt"].std():.4f}')
print(f'    Normal transactions    : {normal_count:,}')
print(f'    Fraudulent transactions: {fraudulent_count:,}')

print('\n(b) Graphs generated (see above cells)')
print('    - Normalized frequency by category')
print('    - is_fraud distribution pie + bar chart')

print('\n(c) Confusion Matrix:')
print(f'    TP={TP:,}  FP={FP:,}  TN={TN:,}  FN={FN:,}')
print(f'    Precision : {precision:.4f} ({precision*100:.2f}%)')
print(f'    Recall    : {recall:.4f} ({recall*100:.2f}%)')
print(f'    Accuracy  : {accuracy:.4f} ({accuracy*100:.2f}%)')
print(f'    F1 Score  : {f1:.4f} ({f1*100:.2f}%)')

print('\n(d) Transaction Prediction:')
print(f'    Jason Johnson | $1,318.89 | shopping_pos | Craig, AK')
print(f'    Predicted: {"FRAUDULENT 🚨" if prediction==1 else "NORMAL ✅"}')
print(f'    Ground Truth: FRAUDULENT (is_fraud = 1)')
print('=' * 60)

---
## Download Output Graphs

In [ ]:
# Download all generated graphs
import os
for fname in ['graph_category_fraud_frequency.png',
              'graph_isfraud_distribution.png',
              'graph_confusion_matrix.png',
              'graph_feature_weights.png']:
    if os.path.exists(fname):
        files.download(fname)
        print(f'Downloaded: {fname}')